# 2025.12 법인 지속거래약화 운영 모델

최종 채택 모델인 `Y_INTERVENE_M12_v2`용 LightGBM과 Isotonic 확률 보정만 사용한다. 2025년 12월 기준 3,372개 법인을 모두 점수화하며 상위 10% 필터는 적용하지 않는다.

In [1]:
from pathlib import Path
import math
import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src').is_dir() and (path / 'outputs').is_dir())
OUTPUT_DIR = ROOT / 'outputs' / '수익성F_outputs'
MODEL_DIR = ROOT / 'outputs' / '수익성F_models'

TARGET = 'Y_INTERVENE_M12_v2'
FEATURE_SET = 'FS2_R1_DACK_DYNAMIC'
CUTOFF_MONTH = 202512
EXPECTED_SCORING_FIRMS = 3341
RANDOM_STATE = 20260718
RETRAIN_FINAL_MODEL = False

DATASET_PATH = OUTPUT_DIR / 'model_m12_intervene_v2_dataset.csv'
FEATURE_SET_PATH = OUTPUT_DIR / 'model_m12_intervene_v2_feature_registry.csv'
OPERATING_FEATURE_PATH = OUTPUT_DIR / 'model_m12_intervene_v2_operating_features_202512.csv'
LOCKED_MODEL_PATH = MODEL_DIR / 'model_m12_intervene_v2_operating_pipeline_202512.joblib'
LOCKED_CALIBRATOR_PATH = MODEL_DIR / 'model_m12_intervene_v2_operating_calibrator_202512.joblib'
WEB_BUNDLE_PATH = MODEL_DIR / 'web_m12_intervene_v2_202512_bundle.joblib'
WEB_SCORE_PATH = ROOT / 'src' / 'models' / 'web_m12_intervene_v2_scores_202512_eligible_3341.csv'
OVERVIEW_TREND_PATH = ROOT / 'src' / 'models' / 'web_m12_overview_risk_trend_202507_202512.csv'

FINAL_PARAMS = {
    'n_estimators': 220,
    'learning_rate': 0.024068490013144556,
    'num_leaves': 22,
    'max_depth': 6,
    'min_child_samples': 85,
    'subsample': 0.9978564144159624,
    'colsample_bytree': 0.6264619236335867,
    'reg_alpha': 0.001889859045062956,
    'reg_lambda': 9.394600689281587,
}

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
registry = pd.read_csv(FEATURE_SET_PATH, encoding='utf-8-sig')
features = registry['feature'].tolist()
if len(features) != 56:
    raise RuntimeError(f'Expected 56 features, found {len(features)}')

operating = pd.read_csv(
    OPERATING_FEATURE_PATH,
    encoding='utf-8-sig',
    dtype={'법인ID': 'string'},
    low_memory=False,
)
operating['법인ID'] = operating['법인ID'].astype(str)
if len(operating) != EXPECTED_SCORING_FIRMS or operating['법인ID'].nunique() != EXPECTED_SCORING_FIRMS:
    raise RuntimeError(f'2025.12 eligible operating data must contain exactly {EXPECTED_SCORING_FIRMS:,} firms.')
if operating['기준월'].astype(str).nunique() != 1 or str(operating['기준월'].iloc[0]) != '2025-12':
    raise RuntimeError('Unexpected operating cutoff month.')
missing_features = sorted(set(features) - set(operating.columns))
if missing_features:
    raise KeyError(f'Missing model features: {missing_features}')

print(f'feature_count={len(features):,}')
print(f'scoring_firms={len(operating):,}')
print(f'y_formula_eligible={len(operating):,}')
print('y_formula_ineligible=0 (excluded before scoring)')

feature_count=56
scoring_firms=3,341
y_formula_eligible=3,341
y_formula_ineligible=0 (excluded before scoring)


In [3]:
def build_final_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        [('numeric', SimpleImputer(strategy='median'), slice(0, None))],
        remainder='drop',
    )
    model = LGBMClassifier(
        **FINAL_PARAMS,
        objective='binary',
        class_weight='balanced',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=-1,
    )
    return Pipeline([('preprocessor', preprocessor), ('model', model)])


def fit_operating_model(train: pd.DataFrame, feature_names: list[str]):
    y = train[TARGET].to_numpy(dtype=int)
    groups = train['법인ID'].astype(str).to_numpy()
    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE + 101,
    )
    raw_oof = np.full(len(train), np.nan, dtype=float)
    for fit_index, valid_index in cv.split(train[feature_names], y, groups):
        fold_model = build_final_pipeline()
        fold_model.fit(train.iloc[fit_index][feature_names], y[fit_index])
        raw_oof[valid_index] = fold_model.predict_proba(
            train.iloc[valid_index][feature_names]
        )[:, 1]
    if not np.isfinite(raw_oof).all():
        raise RuntimeError('OOF probabilities are incomplete.')
    calibrator = IsotonicRegression(out_of_bounds='clip')
    calibrator.fit(raw_oof, y)
    final_model = build_final_pipeline()
    final_model.fit(train[feature_names], y)
    return final_model, calibrator


if RETRAIN_FINAL_MODEL:
    labeled = pd.read_csv(
        DATASET_PATH,
        encoding='utf-8-sig',
        dtype={'법인ID': 'string'},
        low_memory=False,
    )
    labeled['법인ID'] = labeled['법인ID'].astype(str)
    pipeline, calibrator = fit_operating_model(labeled, features)
    joblib.dump(pipeline, LOCKED_MODEL_PATH)
    joblib.dump(calibrator, LOCKED_CALIBRATOR_PATH)
else:
    pipeline = joblib.load(LOCKED_MODEL_PATH)
    calibrator = joblib.load(LOCKED_CALIBRATOR_PATH)

bundle = {
    'pipeline': pipeline,
    'calibrator': calibrator,
    'features': features,
    'target': TARGET,
    'feature_set': FEATURE_SET,
    'cutoff_month': CUTOFF_MONTH,
    'calibration': 'ISOTONIC',
}
joblib.dump(bundle, WEB_BUNDLE_PATH)
print(f'web_bundle={WEB_BUNDLE_PATH}')

web_bundle=/Users/gggyyu/Final_project/outputs/수익성F_models/web_m12_intervene_v2_202512_bundle.joblib


In [4]:
def relative_change_pct(log_difference: pd.Series) -> np.ndarray:
    values = pd.to_numeric(log_difference, errors='coerce').to_numpy(dtype=float)
    return 100.0 * np.expm1(values)


def add_local_shap_top10(frame: pd.DataFrame, result: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    model_pipeline = bundle['pipeline']
    feature_names = bundle['features']
    if hasattr(model_pipeline, 'named_steps'):
        transformed = model_pipeline.named_steps['preprocessor'].transform(frame[feature_names])
        contributions = model_pipeline.named_steps['model'].predict(
            transformed, pred_contrib=True
        )
    else:
        contributions = model_pipeline.booster_.predict(frame[feature_names], pred_contrib=True)
    contributions = np.asarray(contributions)[:, :len(feature_names)]
    top_index = np.argsort(-np.abs(contributions), axis=1)[:, :10]
    for rank in range(10):
        index = top_index[:, rank]
        result[f'shap_top{rank + 1}_feature'] = [feature_names[i] for i in index]
        result[f'shap_top{rank + 1}_value'] = contributions[np.arange(len(frame)), index]
    return result


def score_all_firms_202512(frame: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    feature_names = bundle['features']
    missing = sorted(set(feature_names) - set(frame.columns))
    if missing:
        raise KeyError(f'Missing model features: {missing}')
    raw_probability = bundle['pipeline'].predict_proba(frame[feature_names])[:, 1]
    risk_probability = bundle['calibrator'].predict(raw_probability)

    result = frame[[
        '법인ID', 'baseline_segment_2023', 'current_segment', 'segment_transition',
        '업종_대분류', '업종_중분류',
    ]].copy().reset_index(drop=True).rename(columns={
        'baseline_segment_2023': 'SEG__baseline_segment_2023',
        'current_segment': 'SEG__current_segment',
        'segment_transition': 'SEG__transition',
        '업종_대분류': 'CTX__업종_대분류__현재',
        '업종_중분류': 'CTX__업종_중분류__현재',
    })
    result['cutoff_month'] = CUTOFF_MONTH
    result['score_eligible'] = True
    result['raw_probability'] = raw_probability
    result['risk_probability'] = risk_probability
    result['risk_probability_pct'] = 100.0 * risk_probability
    result['risk_rank_eligible_3341'] = (
        pd.Series(risk_probability).rank(method='first', ascending=False).astype(int)
    )
    result['risk_percentile'] = 100.0 * (
        len(result) - result['risk_rank_eligible_3341'] + 1
    ) / len(result)

    for source, label in {
        '요구불': '요구불_최근3대이전9_변화율_pct',
        '자동이체': '자동이체_최근3대이전9_변화율_pct',
        '채널': '채널_최근3대이전9_변화율_pct',
        '카드': '카드_최근3대이전9_변화율_pct',
    }.items():
        result[label] = relative_change_pct(
            frame[f'{source}_최근3대이전9_로그차이']
        )

    result = add_local_shap_top10(frame, result, bundle)
    return result.sort_values(
        ['risk_rank_eligible_3341', '법인ID'], kind='mergesort'
    ).reset_index(drop=True)


web_scores = score_all_firms_202512(operating, bundle)
web_scores.to_csv(WEB_SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'output={WEB_SCORE_PATH}')
print(f'rows={len(web_scores):,}')
print(f'unique_firms={web_scores["법인ID"].nunique():,}')
web_scores.head(10)

output=/Users/gggyyu/Final_project/src/models/web_m12_intervene_v2_scores_202512_eligible_3341.csv
rows=3,341
unique_firms=3,341


,법인ID,SEG__baseline_segment_2023,SEG__current_segment,SEG__transition,CTX__업종_대분류__현재,CTX__업종_중분류__현재,cutoff_month,score_eligible,raw_probability,risk_probability,...,shap_top6_feature,shap_top6_value,shap_top7_feature,shap_top7_value,shap_top8_feature,shap_top8_value,shap_top9_feature,shap_top9_value,shap_top10_feature,shap_top10_value
0,ae78ab5b2ff86031aab3d3f6438c81497cda9560754c55...,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,도매 및 소매업,소매업; 자동차 제외,202512,True,0.972594,0.891173,...,자동이체_H2대H1_로그차이,0.244741,자동이체_최근3대이전9_로그차이,0.170056,요구불_H2대H1_로그차이,0.159837,요구불_로그변동성,0.142353,자동이체_현재월대직전6_로그차이,0.113476
1,0a77628abb8c5baeb68b65a896daf9a38e27b90bdc4ec3...,거래·수신중심형,저거래·저수신형,거래·수신중심형 → 저거래·저수신형,제조업,기타 기계 및 장비 제조업,202512,True,0.970798,0.857143,...,요구불_최근3대이전9_로그차이,0.230937,카드_최근3대이전9_활성률차이,0.184531,요구불_H2대H1_로그차이,0.165280,자동이체_최근3대이전9_로그차이,0.150224,요구불_로그변동성,0.149227
2,be4c559f18493bab29c7f77bb1034e6dc884dd402509f8...,복합고관계형,복합고관계형,복합고관계형 → 복합고관계형,도매 및 소매업,도매 및 상품 중개업,202512,True,0.971303,0.857143,...,자동이체_H2대H1_로그차이,0.218537,요구불_H2대H1_로그차이,0.164174,요구불_로그변동성,0.155306,카드_최근3대이전9_활성률차이,0.149126,자동이체_최근3대이전9_로그차이,0.125564
3,db697eb8f79eb71c22a76a9156429deba0cbd961ac1cef...,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,제조업,"전자부품, 컴퓨터, 영상, 음향 및 통신장비 제조업",202512,True,0.970796,0.857143,...,요구불_최근3대이전9_로그차이,0.230762,자동이체_최근3대이전9_로그차이,0.173965,요구불_로그변동성,0.163867,요구불_H2대H1_로그차이,0.160677,카드_최근3대이전9_활성률차이,0.155492
4,0f4f0c4b9942d0bd7678d86cf9ea47fd44dbcf6d650edb...,복합고관계형,저거래·저수신형,복합고관계형 → 저거래·저수신형,도매 및 소매업,도매 및 상품 중개업,202512,True,0.964260,0.845070,...,자동이체_H2대H1_로그차이,0.203854,요구불_로그변동성,0.149998,카드_최근3대이전9_활성률차이,0.148518,채널_로그변동성,0.117237,채널_활성폭,0.111857
5,409a17ccf9aa04a06cec3294ace29bb542b9b2f574b6a7...,거래·수신중심형,저거래·저수신형,거래·수신중심형 → 저거래·저수신형,건설업,전문직별 공사업,202512,True,0.964321,0.845070,...,요구불_H2대H1_로그차이,0.206872,요구불_로그변동성,0.172642,자동이체_최근3대이전9_로그차이,0.170498,카드_최근3대이전9_활성률차이,0.153796,채널_활성폭,0.144266
6,489493424e4301baefd4bc90914c72b848d3e95b746f1a...,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,도매 및 소매업,소매업; 자동차 제외,202512,True,0.964258,0.845070,...,요구불_H2대H1_로그차이,0.224462,요구불_로그변동성,0.205382,자동이체_최근3대이전9_로그차이,0.164201,카드_최근3대이전9_활성률차이,0.159754,채널_로그변동성,0.133991
7,685d3200b2e5de34133fe93eae056e3c85e1e0f34bbee7...,복합고관계형,저거래·저수신형,복합고관계형 → 저거래·저수신형,제조업,목재 및 나무제품 제조업; 가구 제외,202512,True,0.969980,0.845070,...,요구불_최근3대이전9_로그차이,0.228451,자동이체_최근3대이전9_로그차이,0.184958,요구불_H2대H1_로그차이,0.163229,요구불_로그변동성,0.162126,채널_활성폭,0.140390
8,722835e512acd998274d62e431d24188dc147da85f0b2b...,복합고관계형,저거래·저수신형,복합고관계형 → 저거래·저수신형,"수도, 하수 및 폐기물 처리, 원료 재생업","폐기물 수집, 운반, 처리 및 원료 재생업",202512,True,0.964632,0.845070,...,요구불_최근3대이전9_로그차이,0.239741,자동이체_최근3대이전9_로그차이,0.160846,채널_활성폭,-0.152032,요구불_H2대H1_로그차이,0.149446,요구불_로그변동성,0.139091
9,96bbb1c9e6f91c91639f7939169932234bfa7f03a66f38...,저거래·저수신형,저거래·저수신형,저거래·저수신형 → 저거래·저수신형,도매 및 소매업,소매업; 자동차 제외,202512,True,0.966944,0.845070,...,자동이체_H2대H1_로그차이,0.229398,요구불_로그변동성,0.175456,자동이체_최근3대이전9_로그차이,0.125793,채널_로그변동성,0.114468,여신관계_수준,0.111233


In [5]:
assert len(web_scores) == EXPECTED_SCORING_FIRMS
assert web_scores['법인ID'].nunique() == EXPECTED_SCORING_FIRMS
assert web_scores['cutoff_month'].eq(202512).all()
assert np.isfinite(web_scores['risk_probability']).all()
assert web_scores['risk_rank_eligible_3341'].min() == 1
assert web_scores['risk_rank_eligible_3341'].max() == EXPECTED_SCORING_FIRMS

web_scores[[
    'risk_probability',
    'risk_probability_pct',
    'risk_rank_eligible_3341',
    'risk_percentile',
]].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

overview_trend = pd.read_csv(OVERVIEW_TREND_PATH, encoding='utf-8-sig')
expected_trend_columns = [
    'as_of_month', 'eligible_count', 'average_risk',
    'high_risk_count', 'high_risk_share', 'model_name',
]
assert overview_trend.columns.tolist() == expected_trend_columns
assert overview_trend['as_of_month'].tolist() == [
    '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12'
]
assert overview_trend['as_of_month'].is_unique
assert overview_trend['average_risk'].between(0, 1).all()
assert overview_trend['high_risk_share'].between(0, 1).all()

december_trend = overview_trend.loc[overview_trend['as_of_month'].eq('2025-12')].iloc[0]
december_high_risk = web_scores['risk_probability'].ge(0.75)
assert int(december_trend['eligible_count']) == len(web_scores) == 3341
assert int(december_trend['high_risk_count']) == int(december_high_risk.sum())
np.testing.assert_allclose(
    december_trend['average_risk'], web_scores['risk_probability'].mean(),
    rtol=0, atol=1e-12,
)
np.testing.assert_allclose(
    december_trend['high_risk_share'], december_high_risk.mean(),
    rtol=0, atol=1e-12,
)
display(overview_trend)


,risk_probability,risk_probability_pct,risk_rank_eligible_3341,risk_percentile
count,3341.000000,3341.000000,3341.000000,3341.000000
mean,0.034610,3.461043,1671.000000,50.014966
std,0.093822,9.382245,964.607951,28.871833
min,0.000000,0.000000,1.000000,0.029931
50%,0.006753,0.675338,1671.000000,50.014966
90%,0.073363,7.336343,3007.000000,90.002993
95%,0.178571,17.857143,3174.000000,95.001497
99%,0.480263,48.026316,3307.600000,99.000299
max,0.891173,89.117267,3341.000000,100.000000
